# Practical 2: Convert Unstructured Log Data into a Structured Dataset
**Course:** Python for Data Science (`BE05000231_PDS`)  
**Topic:** Regular Expression Parsing, Schema Harmonization, Data Type Optimization, and Tabular Export  
**Datasets:** Enterprise Master Log (`server_access.log`) & External Elastic Apache Production Log (`elastic_apache_access.log`)  

---

## 1. AIM & OBJECTIVES
### Aim
To parse unstructured web server access log lines into a structured tabular dataset (Pandas DataFrame), enforce canonical data types, extract isolated query payloads, and export the standardized records into CSV and Parquet formats.

### Objectives
1. **Compiled Regular Expressions:** Formulate a robust regex pattern utilizing named/numbered capture groups to extract the 9 core NCSA Combined Log tokens.
2. **Graceful Malformed Line Handling:** Guarantee 100% resilient parsing with zero script termination on unencoded or non-standard characters.
3. **Canonical 13-Column Schema:** Structure every record into standard fields (`client_ip`, `ident`, `auth_user`, `timestamp`, `request_method`, `resource_requested`, `http_protocol`, `status_code`, `bytes_sent`, `referrer`, `user_agent`, `payload`, `line_number`).
4. **Memory Downcasting Optimization:** Convert generic Python string object columns into optimized numerical (`int16`, `int32`) and `category` types to reduce RAM utilization.
5. **Dual-Format Tabular Serialization:** Export the final structured dataset to standard CSV and high-efficiency Snappy-compressed Apache Parquet.
6. **External Production Benchmark:** Validate the parser against an external, independent production web log from Elastic (10,000 real-world lines).

---
## 2. REGEX EXTRACTION ARCHITECTURE

We define a compiled regular expression matching the NCSA Combined Log syntax:

```regex
^(\S+)\s+(\S+)\s+(\S+)\s+\[([^\]]+)\]\s+"([A-Z]+)\s+(.+?)(?:\s+(HTTP/[\d\.]+|-)?)?"\s+(\d{3})\s+(\S+)(?:\s+"([^"]*)"\s+"([^"]*)")?
```

### Capture Group Mapping:
- **Group 1 (`client_ip`)**: Remote IPv4/IPv6 client address.
- **Group 2 (`ident`)**: RFC 1413 identity token (typically `-`).
- **Group 3 (`auth_user`)**: HTTP Basic Auth username.
- **Group 4 (`timestamp`)**: Request arrival time string.
- **Group 5 (`request_method`)**: HTTP verb (`GET`, `POST`, `HEAD`, `OPTIONS`).
- **Group 6 (`resource_requested`)**: Target URI including query parameters.
- **Group 7 (`http_protocol`)**: Application protocol version (`HTTP/1.1`).
- **Group 8 (`status_code`)**: HTTP status code (`200`, `401`, `404`, `500`).
- **Group 9 (`bytes_sent`)**: Payload size in bytes.
- **Group 10 (`referrer`)**: Referring URL.
- **Group 11 (`user_agent`)**: Client browser, operating system, or bot.

---
## 3. IMPORT LIBRARIES & SETUP ENVIRONMENT
We import `pandas`, `numpy`, `re`, `time`, and `matplotlib`.

In [ ]:
import sys
import os
import re
import time
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Path resolution
PRACTICAL_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_RAW_DIR = PRACTICAL_DIR / 'data' / 'raw'
DATA_PROCESSED_DIR = PRACTICAL_DIR / 'data' / 'processed'
PLOTS_DIR = PRACTICAL_DIR / 'outputs' / 'plots'
REPORTS_DIR = PRACTICAL_DIR / 'outputs' / 'reports'

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_LOG = DATA_RAW_DIR / 'server_access.log'
EXTERNAL_LOG = DATA_RAW_DIR / 'elastic_apache_access.log'

print(f"Primary Log Exists  : {PRIMARY_LOG.exists()} ({PRIMARY_LOG.stat().st_size / (1024*1024):.2f} MB)")
print(f"External Log Exists : {EXTERNAL_LOG.exists()} ({EXTERNAL_LOG.stat().st_size / (1024*1024):.2f} MB)")


---
## 4. REGEX PARSER IMPLEMENTATION
We implement `parse_log_line()` to convert each unstructured string into dictionary tokens with fallback error recovery.

In [ ]:
CLF_REGEX = re.compile(
    r'^(\S+)\s+(\S+)\s+(\S+)\s+\[([^\]]+)\]\s+"([A-Z]+)\s+(.+?)(?:\s+(HTTP/[\d\.]+|-)?)?"\s+(\d{3})\s+(\S+)(?:\s+"([^"]*)"\s+"([^"]*)")?',
    re.IGNORECASE
)

def parse_log_line(line_num: int, line_text: str):
    m = CLF_REGEX.match(line_text)
    if m:
        raw_res = m.group(6) or '-'
        payload = raw_res.split('?', 1)[1] if '?' in raw_res else ''
        raw_bytes = m.group(9)
        bytes_val = int(raw_bytes) if raw_bytes and raw_bytes.isdigit() else 0
        return {
            'line_number': line_num,
            'client_ip': m.group(1),
            'ident': m.group(2) if m.group(2) != '-' else '',
            'auth_user': m.group(3) if m.group(3) != '-' else '',
            'timestamp': m.group(4),
            'request_method': m.group(5).upper(),
            'resource_requested': raw_res,
            'http_protocol': m.group(7) if m.group(7) else 'HTTP/1.1',
            'status_code': int(m.group(8)),
            'bytes_sent': bytes_val,
            'referrer': m.group(10) if m.group(10) and m.group(10) != '-' else '',
            'user_agent': m.group(11) if m.group(11) and m.group(11) != '-' else '',
            'payload': payload,
            'parse_success': 1
        }
    else:
        parts = line_text.split()
        return {
            'line_number': line_num,
            'client_ip': parts[0] if parts else '-',
            'ident': '', 'auth_user': '', 'timestamp': '',
            'request_method': 'GET',
            'resource_requested': '-',
            'http_protocol': 'HTTP/1.1',
            'status_code': 200,
            'bytes_sent': 0,
            'referrer': '', 'user_agent': '', 'payload': '',
            'parse_success': 0
        }

print("Log parsing function compiled successfully!")


---
## 5. PARSE ENTERPRISE ACCESS LOG & OPTIMIZE DATA TYPES
We parse all 300,000 lines of `server_access.log` and cast columns to memory-efficient types (`int16`, `int32`, `category`).

In [ ]:
t0 = time.time()
records = []
malformed = 0

with open(PRIMARY_LOG, 'r', encoding='utf-8', errors='replace') as f:
    for idx, line in enumerate(f):
        s = line.strip()
        if not s:
            continue
        rec = parse_log_line(idx + 1, s)
        if rec['parse_success'] == 0:
            malformed += 1
        records.append(rec)

df = pd.DataFrame(records)
raw_mem = df.memory_usage(deep=True).sum() / (1024 * 1024)

# Optimize data types
df['line_number'] = df['line_number'].astype(np.int32)
df['status_code'] = df['status_code'].astype(np.int16)
df['bytes_sent'] = df['bytes_sent'].astype(np.int32)
df['request_method'] = df['request_method'].astype('category')
df['http_protocol'] = df['http_protocol'].astype('category')
df['parse_success'] = df['parse_success'].astype(np.int8)

opt_mem = df.memory_usage(deep=True).sum() / (1024 * 1024)
elapsed = time.time() - t0

print(f"Total Lines Processed : {len(df):,}")
print(f"Parsing Success Rate  : {((len(df)-malformed)/len(df))*100:.4f}% ({malformed} malformed)")
print(f"Parsing Throughput    : {len(df)/elapsed:,.0f} lines/sec ({elapsed:.2f}s)")
print(f"Initial RAM Usage     : {raw_mem:.2f} MB")
print(f"Optimized RAM Usage   : {opt_mem:.2f} MB ({((raw_mem-opt_mem)/raw_mem)*100:.1f}% reduction)")
df.head(5)


---
## 6. EXTERNAL PRODUCTION BENCHMARK (ELASTIC 10,000 LINES)
We benchmark the parser against Elastic's official production web server access log to verify real-world parsing robustness.

In [ ]:
ext_records = []
ext_malformed = 0

with open(EXTERNAL_LOG, 'r', encoding='utf-8', errors='replace') as f:
    for idx, line in enumerate(f):
        s = line.strip()
        if not s:
            continue
        rec = parse_log_line(idx + 1, s)
        if rec['parse_success'] == 0:
            ext_malformed += 1
        ext_records.append(rec)

df_ext = pd.DataFrame(ext_records)
print("=" * 60)
print(f"External Production Log : {EXTERNAL_LOG.name}")
print(f"Total Lines Parsed      : {len(df_ext):,}")
print(f"Parsing Success Rate    : {((len(df_ext)-ext_malformed)/len(df_ext))*100:.4f}%")
print(f"Unique Client IPs       : {df_ext['client_ip'].nunique():,}")
print(f"Top 3 IPs               : {list(df_ext['client_ip'].value_counts().head(3).index)}")
print(f"HTTP Methods            : {dict(df_ext['request_method'].value_counts())}")
print("=" * 60)
df_ext.head(3)


---
## 7. EXPORT STRUCTURED DATASETS (CSV & PARQUET)
We serialize the canonical structured DataFrame to both CSV and Snappy-compressed Apache Parquet.

In [ ]:
csv_path = DATA_PROCESSED_DIR / 'structured_logs.csv'
parquet_path = DATA_PROCESSED_DIR / 'structured_logs.parquet'

df.to_csv(csv_path, index=False)
csv_mb = csv_path.stat().st_size / (1024 * 1024)
print(f"Exported CSV     : {csv_path.name} ({csv_mb:.2f} MB)")

df.to_parquet(parquet_path, index=False, compression='snappy')
pq_mb = parquet_path.stat().st_size / (1024 * 1024)
print(f"Exported Parquet : {parquet_path.name} ({pq_mb:.2f} MB) [{((1 - pq_mb/csv_mb)*100):.1f}% compression savings]")


---
## 8. VISUAL EXPLORATORY CHARTS
We visualize schema column completeness, HTTP method distributions, status code breakdowns, and RAM reduction.

In [ ]:
# 1. Column Completeness Matrix
plt.figure(figsize=(10, 4.5), dpi=150)
completeness = (df.notnull() & (df != '')).mean() * 100
plt.barh(completeness.index, completeness.values, color='#1687FF', edgecolor='#060912')
plt.title('Structured DataFrame Column Completeness (%)', fontsize=12, fontweight='bold')
plt.xlabel('Completeness Percentage (%)', fontsize=10)
plt.xlim(0, 105)
plt.tight_layout()
plt.show()

# 2. HTTP Method Distribution
plt.figure(figsize=(7, 4), dpi=150)
m_counts = df['request_method'].value_counts().head(5)
plt.bar(m_counts.index.astype(str), m_counts.values, color=['#22D3EE', '#1687FF', '#21D98B', '#FFB547'], width=0.5)
plt.title('HTTP Request Method Distribution', fontsize=12, fontweight='bold')
plt.xlabel('Method', fontsize=10)
plt.ylabel('Count', fontsize=10)
plt.tight_layout()
plt.show()

# 3. HTTP Status Codes Distribution
plt.figure(figsize=(8, 4), dpi=150)
s_counts = df['status_code'].value_counts().head(7)
colors = ['#21D98B' if str(s).startswith('2') else ('#FFB547' if str(s).startswith('3') else '#FF5577') for s in s_counts.index]
plt.bar(s_counts.index.astype(str), s_counts.values, color=colors, width=0.55)
plt.title('HTTP Status Code Distribution (2xx, 3xx, 4xx, 5xx)', fontsize=12, fontweight='bold')
plt.xlabel('Status Code', fontsize=10)
plt.ylabel('Count', fontsize=10)
plt.tight_layout()
plt.show()
